# IndiaData Colab Runner (Clean)
Use this notebook in Colab with Python 3 runtime.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev libfontconfig1-dev git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/abhinavjnu/IndiaData-colab.git'
!rm -rf /content/IndiaData
!git clone "$REPO_URL" /content/IndiaData
%cd /content/IndiaData
!git branch --show-current

In [ ]:
import os
drive_root = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else '/content/drive/My Drive'
base = f"{drive_root}/IndiaDataData"
for p in ['raw', 'processed', 'codebooks', 'outputs/tables', 'outputs/figures', 'logs']:
    os.makedirs(f"{base}/{p}", exist_ok=True)
print(base)

In [ ]:
import textwrap
config_text = textwrap.dedent(f'''
api:
  base_url: "https://www.microdata.gov.in/NADA/index.php/api/"
  api_key: "REPLACE_WITH_YOUR_API_KEY"

paths:
  raw_data: "{base}/raw"
  processed_data: "{base}/processed"
  codebooks: "{base}/codebooks"
  tables: "{base}/outputs/tables"
  figures: "{base}/outputs/figures"

settings:
  default_survey: "plfs"
  save_format: "parquet"

surveys:
  plfs:
    source: "microdata.gov.in"
''').strip() + '\n'
with open('/content/IndiaData/config.yaml', 'w', encoding='utf-8') as f:
    f.write(config_text)
print('config written')

In [ ]:
!Rscript "R/00_setup.R"
!Rscript -e 'source("R/01_config.R"); print_config()'

In [ ]:
!Rscript "PLFS/automated_plfs_analysis.R" 2>&1 | tee "{base}/logs/plfs_run.log"

In [ ]:
!ls -lah "{base}/logs"
!ls -lah "{base}/outputs/tables"
!ls -lah "{base}/outputs/figures"